# Árbol de decisión sin preprocesamiento

In [43]:
# Modelo
from sklearn.tree import DecisionTreeRegressor

# Entrenamiento
from sklearn.model_selection import train_test_split, GridSearchCV

# Métricas
from sklearn.metrics import plot_roc_curve, classification_report, plot_confusion_matrix, roc_auc_score

# Gráficos
from matplotlib import pyplot as plt
import seaborn as sns
import graphviz
import dtreeviz.trees as dtreeviz

# Otros
import pandas as pd
import numpy as np

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [44]:
%%R
source("Utils.R")

#setwd("C:/foo")

setSeed()

vasijas_X <- obtener_X()
vasijas_Y <- obtener_Y()

df <- vasijas_X
df$Y <- vasijas_Y

hold_out_ind <- obtener_holdout_ind(vasijas_X)

HOLDOUT_X <- vasijas_X[hold_out_ind, ]
HOLDOUT_Y <- vasijas_Y[hold_out_ind]
HOLDOUT_DF <- df[hold_out_ind, ]

TRAIN_Y <- vasijas_Y[-hold_out_ind]
TRAIN_X <- vasijas_X[-hold_out_ind, ]
TRAIN_DF <- df[-hold_out_ind, ]

write.csv(TRAIN_X, "/home/lukas/FIUBA/AE/Repositorio/trabajo_practico3/src/TRAIN_X/tmp.csv")
write.csv(TRAIN_Y, "/home/lukas/FIUBA/AE/Repositorio/trabajo_practico3/src/TRAIN_Y/tmp.csv")

In [45]:
TRAIN_X = pd.read_csv("TRAIN_X/tmp.csv")
TRAIN_Y = pd.read_csv("TRAIN_Y/tmp.csv")
del TRAIN_X['Unnamed: 0']
del TRAIN_Y['Unnamed: 0']

In [48]:
params = {
    'criterion': ['mse', 'friedman_mse', 'mae', 'poisson'],
    'splitter': ['best'],
    'max_depth': range(5,22,1),
    'min_samples_leaf': np.linspace(0.0001, 0.001, 10),
    'max_leaf_nodes': [2**x for x in range(6, 18, 2)],
    'min_impurity_decrease': np.linspace(0, 0.1, 5)
}
model = DecisionTreeRegressor()

gscv_arbol = GridSearchCV(model, params, scoring='r2', n_jobs=6, cv=10, verbose=4)

In [50]:
gscv_arbol.fit(TRAIN_X, TRAIN_Y)

Fitting 10 folds for each of 20400 candidates, totalling 204000 fits


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:    1.2s
[Parallel(n_jobs=6)]: Done 194 tasks      | elapsed:    2.6s
[Parallel(n_jobs=6)]: Done 686 tasks      | elapsed:    5.9s
[Parallel(n_jobs=6)]: Done 1370 tasks      | elapsed:   11.4s
[Parallel(n_jobs=6)]: Done 2254 tasks      | elapsed:   18.3s
[Parallel(n_jobs=6)]: Done 3330 tasks      | elapsed:   26.4s
[Parallel(n_jobs=6)]: Done 4606 tasks      | elapsed:   36.5s
[Parallel(n_jobs=6)]: Done 6074 tasks      | elapsed:   47.7s
[Parallel(n_jobs=6)]: Done 7742 tasks      | elapsed:  1.0min
[Parallel(n_jobs=6)]: Done 9602 tasks      | elapsed:  1.3min
[Parallel(n_jobs=6)]: Done 11662 tasks      | elapsed:  1.5min
[Parallel(n_jobs=6)]: Done 13914 tasks      | elapsed:  1.8min
[Parallel(n_jobs=6)]: Done 16366 tasks      | elapsed:  2.1min
[Parallel(n_jobs=6)]: Done 19010 tasks      | elapsed:  2.5min
[Parallel(n_jobs=6)]: Done 21854 tasks      | elapsed:  

GridSearchCV(cv=10, estimator=DecisionTreeRegressor(), n_jobs=6,
             param_grid={'criterion': ['mse', 'friedman_mse', 'mae', 'poisson'],
                         'max_depth': range(5, 22),
                         'max_leaf_nodes': [64, 256, 1024, 4096, 16384, 65536],
                         'min_impurity_decrease': array([0.   , 0.025, 0.05 , 0.075, 0.1  ]),
                         'min_samples_leaf': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009, 0.001 ]),
                         'splitter': ['best']},
             scoring='r2', verbose=4)

In [52]:
print(gscv_arbol.best_estimator_)

DecisionTreeRegressor(criterion='mae', max_depth=5, max_leaf_nodes=256,
                      min_samples_leaf=0.0009)


In [53]:
print(gscv_arbol.best_score_)

0.29766646956328036
